## 10. Synthesis: runtime, throughput, and footprint

Four tools solved the same 1D bump-pulse step: JAX, PyOMP, nanobind, and CppJIT. Each was measured against the NumPy baseline at `N = 16384` over `1000` steps. This notebook reads `timings.json` and compares them on runtime, throughput, and working-set footprint, then closes with a fair float64 comparison of the two GPU paths.

### Table of Contents

1. [Read the timings](#sec1)
2. [Runtime: warm and cold](#sec2)
3. [Throughput: cells/s and effective bandwidth](#sec3)
4. [Memory footprint](#sec4)
5. [Matched-precision GPU comparison](#sec5)
6. [Reference: which tool when](#sec6)

### <a id="sec1"></a>1. Read the timings

We load every row from `timings.json` (written by NB 05-09). Each row
carries the warm wall-clock (`median_s`), the first-call cost where the
tool JIT-compiles (`cold_s`), the hardware and dtype it ran on, and the
max-diff against the float64 NumPy reference.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import swe_core

rows = swe_core.load_timings()
by_tool = {r["tool"]: r for r in rows}

# timings.json is written by notebooks 05 to 09
expected = ["numpy", "jax", "pyomp", "nanobind", "cppjit_gpu_thrust"]
missing = [t for t in expected if t not in by_tool]
if missing:
    raise SystemExit(
        "timings.json has no rows for: " + ", ".join(missing) + ". "
        "Run notebooks 05 through 09 first, then re-run this notebook."
    )

hdr = (f'{"stage":<24}{"tool":>18}{"hardware":>9}{"dtype":>9}'
       f'{"median_s":>11}{"cold_s":>9}{"max_diff":>12}')
print(hdr)
print("-" * len(hdr))
for r in rows:
    cold = r.get("cold_s", float("nan"))
    md = r.get("max_diff_vs_numpy")
    md_s = f"{md:.1e}" if md is not None else ("ref" if r["tool"] == "numpy" else "n/a")
    print(f'{r["stage"]:<24}{r["tool"]:>18}{r["hardware"]:>9}{r["dtype"]:>9}'
          f'{r["median_s"]:>11.4f}{cold:>9.3f}{md_s:>12}')

### <a id="sec2"></a>2. Runtime: warm and cold

Warm wall-clock is the steady-state median over repeated runs. Cold time is the first call, where JAX, PyOMP, and CppJIT pay a compile cost before they execute. Each bar is labelled with its hardware and dtype. The runs are not on common ground, because JAX is on the GPU in float32, CppJIT is on the GPU in float64, and the rest are on the CPU in float64. Read the bars one tool at a time rather than as a single ranking.

In [ ]:
# Shared comparison set across the ladder
order  = ["numpy", "jax", "pyomp", "nanobind", "cppjit_gpu_thrust"]
labels = ["NumPy\nCPU fp64", "JAX\nGPU fp32", "PyOMP\nCPU fp64 x8",
          "nanobind\nCPU fp64", "CppJIT\nGPU fp64"]
colors = ["#888", "#27a", "#5a8", "#c33", "#a36"]

warm = [by_tool[t]["median_s"] * 1e3 for t in order]                 # ms
cold = [by_tool[t].get("cold_s", float("nan")) * 1e3 for t in order]  # ms

fig, (ax_w, ax_c) = plt.subplots(1, 2, figsize=(12, 4))

ax_w.bar(labels, warm, color=colors)
ax_w.set_yscale("log")
ax_w.set_ylabel("warm median [ms, log]")
ax_w.set_title("Warm wall-clock (1000 steps, N=16384)")
ax_w.grid(alpha=0.3, axis="y")

# Cold vs warm for the tools that JIT-compile on the first call.
jit = ["jax", "pyomp", "nanobind", "cppjit_gpu_thrust"]
jit_labels = {"jax": "JAX", "pyomp": "PyOMP", "nanobind": "nanobind",
              "cppjit_gpu_thrust": "CppJIT"}
x = np.arange(len(jit)); w = 0.38
ax_c.bar(x - w/2, [by_tool[t]["cold_s"] * 1e3 for t in jit], w, label="cold (first call)", color="#c33")
ax_c.bar(x + w/2, [by_tool[t]["median_s"] * 1e3 for t in jit], w, label="warm (median)", color="#5a8")
ax_c.set_xticks(x); ax_c.set_xticklabels([jit_labels[t] for t in jit])
ax_c.set_yscale("log"); ax_c.set_ylabel("time [ms, log]")
ax_c.set_title("First-call compile cost"); ax_c.legend(); ax_c.grid(alpha=0.3, axis="y")

plt.tight_layout(); plt.show()

### <a id="sec3"></a>3. Throughput: cells per second and effective bandwidth

Cells per second (`N x n_steps / time`) is the primary metric, comparable across grid sizes and step counts. Effective bandwidth (four array passes per step) is a secondary view. Both GPU tools beat the CPU here, where the grid is large enough to keep the device busy. JAX posts the highest throughput, but only because it runs float32. CppJIT runs float64, exact to `1e-15`, and at matched precision is the faster of the two (section 5).

In [ ]:
N, STEPS, PASSES = 16384, 1000, 4
def nbytes(dtype): return 4 if dtype.endswith("32") else 8

cells_s, gb_s = [], []
for t in order:
    r = by_tool[t]; m = r["median_s"]
    cells_s.append(N * STEPS / m)                                  # cell-updates/s
    gb_s.append(PASSES * N * nbytes(r["dtype"]) * STEPS / m / 1e9)  # GB/s (effective)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.bar(labels, [c / 1e6 for c in cells_s], color=colors)
ax1.set_ylabel("Mcell-updates / s"); ax1.set_title("Throughput (higher is better)")
ax1.grid(alpha=0.3, axis="y")
ax2.bar(labels, gb_s, color=colors)
ax2.set_ylabel("effective GB/s (cache-resident)"); ax2.set_title("Effective bandwidth")
ax2.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()

print(f'{"tool":<12}{"Mcells/s":>12}{"eff GB/s":>12}')
for t, c, g in zip(order, cells_s, gb_s):
    tag = "  (placeholder)" if by_tool[t].get("activated") is False else ""
    print(f'  {t:<10}{c/1e6:>12.1f}{g:>12.2f}{tag}')

### <a id="sec4"></a>4. Memory footprint

The solver keeps two conserved fields, `h` and `hu`, double-buffered for the step, so the working set is four arrays of `N + 2` at the tool's dtype. At `N = 16384` that is about 512 KiB in float64 and half that in float32. This is an analytical footprint rather than a measured resident size. It is small enough that memory capacity is never the constraint at this problem size, and dtype is the only axis that moves it. Real resident memory is dominated by framework overhead such as the Python interpreter, libomp, and JAX's pre-allocation of device memory, which we do not show.

In [ ]:
footprint = {}
for t in order:
    dt = by_tool[t]["dtype"]
    footprint[t] = 4 * (N + 2) * nbytes(dt) / 1024.0  # KiB

print(f'{"tool":<12}{"dtype":>8}{"working set":>14}')
for t in order:
    print(f'  {t:<10}{by_tool[t]["dtype"]:>8}{footprint[t]:>11.1f} KiB')

### <a id="sec5"></a>5. Matched-precision GPU comparison

The bars above use each tool's native precision, so the two GPU paths are not directly comparable: JAX runs float32, CppJIT float64. Re-running both at float64 across a range of `N` gives a fair check. On this consumer GPU float64 is throttled, so the absolute numbers are low, but both paths pay that equally.

In [ ]:
# Matched-precision (float64) comparison: NumPy (CPU) vs JAX (lax.scan) vs CppJIT (Thrust).
# A short sweep (200 steps) over the grid size, reporting cell-updates per second.
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import cppjit
assert cppjit.CUDA_ENABLED, 'this comparison needs the CUDA-enabled tutorial image'

DRY = 1e-6
DUEL_STEPS = 200

@jax.jit
def _jax_step(state, _, dx, dt, g):
    h, hu = state
    h  = h.at[0].set(h[1]).at[-1].set(h[-2])
    hu = hu.at[0].set(-hu[1]).at[-1].set(-hu[-2])
    hL, hR = h[:-1], h[1:]; huL, huR = hu[:-1], hu[1:]
    hsL = jnp.maximum(hL, DRY); hsR = jnp.maximum(hR, DRY)
    uL = huL / hsL; uR = huR / hsR
    cL = jnp.sqrt(g * hsL); cR = jnp.sqrt(g * hsR)
    a  = jnp.maximum(jnp.abs(uL) + cL, jnp.abs(uR) + cR)
    F_h  = 0.5 * (huL + huR) - 0.5 * a * (hR - hL)
    F_hu = 0.5 * (huL*uL + 0.5*g*hL*hL + huR*uR + 0.5*g*hR*hR) - 0.5 * a * (huR - huL)
    h_new  = h.at[1:-1].set(h[1:-1]  - (dt/dx) * (F_h[1:]  - F_h[:-1]))
    hu_new = hu.at[1:-1].set(hu[1:-1] - (dt/dx) * (F_hu[1:] - F_hu[:-1]))
    return (h_new, hu_new), None

def run_jax_fp64(n_cells, n_steps, cell_dx, step_dt):
    h, hu = swe_core.bump_ic(n_cells, L=10.0, h0=1.0, amplitude=0.1, sigma=0.5)
    state = (jnp.asarray(h), jnp.asarray(hu))
    final, _ = jax.lax.scan(lambda s, x: _jax_step(s, x, cell_dx, step_dt, 9.81),
                            state, jnp.arange(n_steps))
    h_f, _ = jax.tree_util.tree_map(lambda a: a.block_until_ready(), final)
    return np.asarray(h_f)

cppjit.cppdef(r"""
#include <cuda_runtime.h>
#include <thrust/execution_policy.h>
#include <thrust/for_each.h>
#include <thrust/iterator/counting_iterator.h>
__device__ inline void gpu_rf(double hL,double hR,double huL,double huR,double g,double& Fh,double& Fhu){
  const double D=1e-6; const double a1=hL>D?hL:D,b1=hR>D?hR:D;
  const double uL=huL/a1,uR=huR/b1; const double cL=sqrt(g*a1),cR=sqrt(g*b1);
  const double a=fmax(fabs(uL)+cL,fabs(uR)+cR);
  Fh=0.5*(huL+huR)-0.5*a*(hR-hL);
  Fhu=0.5*(huL*uL+0.5*g*hL*hL+huR*uR+0.5*g*hR*hR)-0.5*a*(huR-huL);
}
void gpu_solve(const double* h0,const double* hu0,double* ho,double* huo,
                long Np2,double dx,double dt,double g,long n){
  const long N=Np2-2; const double inv=dt/dx;
  double *h,*hu,*hn,*hun;
  cudaMallocManaged(&h,Np2*8);cudaMallocManaged(&hu,Np2*8);
  cudaMallocManaged(&hn,Np2*8);cudaMallocManaged(&hun,Np2*8);
  for(long i=0;i<Np2;++i){h[i]=h0[i];hu[i]=hu0[i];}
  for(long s=0;s<n;++s){
    double *H=h,*HU=hu,*HN=hn,*HUN=hun;
    thrust::for_each(thrust::device,thrust::counting_iterator<long>(0),thrust::counting_iterator<long>(1),
      [=]__device__(long){H[0]=H[1];H[Np2-1]=H[Np2-2];HU[0]=-HU[1];HU[Np2-1]=-HU[Np2-2];
        HN[0]=H[0];HUN[0]=HU[0];HN[Np2-1]=H[Np2-1];HUN[Np2-1]=HU[Np2-1];});
    thrust::for_each(thrust::device,thrust::counting_iterator<long>(1),thrust::counting_iterator<long>(N+1),
      [=]__device__(long i){double a,b,c,d;gpu_rf(H[i-1],H[i],HU[i-1],HU[i],g,a,b);
        gpu_rf(H[i],H[i+1],HU[i],HU[i+1],g,c,d);HN[i]=H[i]-inv*(c-a);HUN[i]=HU[i]-inv*(d-b);});
    double*t=h;h=hn;hn=t;t=hu;hu=hun;hun=t;
  }
  cudaDeviceSynchronize();
  for(long i=0;i<Np2;++i){ho[i]=h[i];huo[i]=hu[i];}
  cudaFree(h);cudaFree(hu);cudaFree(hn);cudaFree(hun);
}
""")

def run_cppjit_fp64(n_cells, n_steps, cell_dx, step_dt):
    h, hu = swe_core.bump_ic(n_cells, L=10.0, h0=1.0, amplitude=0.1, sigma=0.5)
    h = np.ascontiguousarray(h); hu = np.ascontiguousarray(hu)
    ho = np.empty_like(h); huo = np.empty_like(hu)
    cppjit.gbl.gpu_solve(h, hu, ho, huo, h.shape[0], cell_dx, step_dt, 9.81, n_steps)
    return ho

def run_numpy_fp64(n_cells, n_steps, cell_dx, step_dt):
    h, hu = swe_core.bump_ic(n_cells, L=10.0, h0=1.0, amplitude=0.1, sigma=0.5)
    for _ in range(n_steps):
        swe_core.apply_bc_reflective(h, hu)
        h, hu = swe_core.step_numpy(h, hu, cell_dx, step_dt, g=9.81)
    return h

DUEL_N = [4096, 16384, 65536, 262144]
rate = {"NumPy CPU": [], "JAX GPU": [], "CppJIT GPU": []}
print(f'{"N":>9}{"NumPy":>10}{"JAX":>10}{"CppJIT":>10}   Mcell/s')
for n_cells in DUEL_N:
    cdx = 10.0 / n_cells
    cdt = swe_core.fixed_dt(1.1, cdx, cfl=0.4, g=9.81)
    rn = swe_core.timed_run(run_numpy_fp64,  n_cells, DUEL_STEPS, cdx, cdt, warmup=0, repeats=1, label="n")
    rj = swe_core.timed_run(run_jax_fp64,    n_cells, DUEL_STEPS, cdx, cdt, warmup=1, repeats=3, label="j")
    rc = swe_core.timed_run(run_cppjit_fp64, n_cells, DUEL_STEPS, cdx, cdt, warmup=1, repeats=3, label="c")
    mc = lambda r: n_cells * DUEL_STEPS / r["median_s"] / 1e6
    rate["NumPy CPU"].append(mc(rn)); rate["JAX GPU"].append(mc(rj)); rate["CppJIT GPU"].append(mc(rc))
    print(f'{n_cells:>9}{mc(rn):>10.0f}{mc(rj):>10.0f}{mc(rc):>10.0f}')

fig, ax = plt.subplots(figsize=(7, 4))
for name, marker in zip(rate, ["s-", "o-", "^-"]):
    ax.plot(DUEL_N, rate[name], marker, label=name)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("N (cells)"); ax.set_ylabel("throughput [Mcell-updates / s]")
ax.set_title("float64 SWE solve: the GPU pulls away as the grid grows")
ax.legend(); ax.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()


### <a id="sec6"></a>6. Reference: which tool when

The numbers above rank by throughput, and the table below
is the qualitative counterpart:

| Tool | Use when | Limitation |
|---|---|---|
| **NumPy** | Prototyping, a single CPU thread is enough | Single-threaded, no GPU |
| **JAX** | Fused GPU program from Python, with autodiff | Shape-rigidity: each new input shape pays a fresh XLA compile |
| **PyOMP** | Portable C-OpenMP semantics (including target offload) inside `@njit` | Thread scaling caps at physical cores/memory bandwidth |
| **nanobind** | Python entry point to existing C++ codebase | Build-system glue and FFI cost, plus a recompile-on-edit cycle |
| **CppJIT** | Automatic Python/C++ bindings, no wrapper code, GPU support | First-call compile latency, high memory utilisation and parsing cost (JIT overhead) with large translation units |